# 08b The 3D slip model: uncertainty, resolution and sensitivity

```
Created on Wed Sep 30 2026
PREPUB: promoted from drafts/pylith/pylith_sfcvm_tests.py and the 2026-09-30 sensitivity runs.
@author: daniellelindsay
Run in opera_disp_feb13, after 08a. About 35 min (about 130 tent inversions of ~10 s; no PyLith run).
```

**What this notebook does.** For the SELECTED model of 08a (aligned SFCVM, nugget Cd), its
companion (07a's full correlated Cd) and the homogeneous PyLith twins of both, on the cached
order-1 Green's functions:

1. **Bootstrap** (07a [11]): noise drawn from the inversion's own Cd (Cholesky) added to the
   model prediction and re-inverted at the selected sigma, N = 07a's `n_boot`; mean and 2-sigma
   of Mw30, Mw(SFCVM mu), potency, surface %, peak slip and its depth. Plus the same numbers at
   the neighbouring grid sigmas (sigma sensitivity).
2. **Synthetic recovery** (07b [11]): 10 mm uniform slip from the surface to D over the SSE
   section (tapered), forward through the PyLith G, noise from the same Cd, inverted with the
   same setup: moment and depth recovery, and the west/east near-field residual (an artefact flag).
3. **Fixed-depth scan** (07b [9]): uniform slip to D, amplitude + constant by whitened least
   squares with the run's Cd, chi2(D), bracket delta chi2 <= 4, Mw per D.
4. **Sensitivity runs**: every other PyLith + CSI inversion of the model selection, read from
   `modeling_SFCVM/gf/` and NOT recomputed here: mesh `shallow` (60 m near-surface band), Cd
   nugget floors 15 and 4 pixels, equal weights (`diagcd`), no surface constraint (`nosurf`),
   swath 7 km, dip 80 E and 70 E, order-2 elements, the SFCVM boundary 50 m east / west of the
   trace (`e50`, `w50`), and the coherence-0.7 field (`coh07`), each with both media and, where
   run, both Cd treatments. Each needs its own Green's-function run, made once with the parameter
   files in `scripts/pylith_cfg/` (the mesh, material and points variants are the cfg files named
   there). The products (`inversion_products<tag>.npz`, `results_comparison<tag>.csv`,
   `lcurve_<medium><tag>.csv`, the table `final_comparison.csv`) are provided. This notebook
   draws the one-page summary of every run and the all-tests figure and copies the table.
Every run is solved at the locked sigma of 08a (`sigma_locked`, Dani 2026-10-03), not at its own
L-curve pick (which each run still computes and records as `sigma_lcurve_pick`), so the spread of the
table is the spread of the factor tested, not of sigma. Each row carries the fingerprint of the 06
field it fits; [5] refuses a table made before the current 06 field or on another field.
PREPUB: The inversions are re-made by `drafts/pylith/models_overnight_2026-10-05.sh` (driver
PREPUB: `drafts/pylith/pylith_sfcvm_gf.py` with its `PYLITH_*` switches, `PYLITH_SIGMA=locked`); the earlier
PREPUB: sets are kept in `modeling_SFCVM/gf/pre_<date>/`.

Surface % is the median modelled slip on the top-edge vertices of the SSE section over the median
of the measured crossings in the SSE section (08a [10]; the one shared denominator of 07a, 08a, this
notebook and the driver, `surface_crossings.surface_pct_denominator_mm`), so cached and fresh
surface % are compared and registered as they are.

08a's cells [1]-[10] are exec'd from `scripts/08a_3d_model.py` (mesh, materials, cached PyLith
G, 07a's data and machinery, the validations, `invert_tents`, `numbers`), so nothing is
re-implemented here beyond the loops; the depth-scan least squares and the window/shape
definitions are those of 07b, applied to the tent (vertex) parameters.
PREPUB: CHECKS against the earlier run: the bootstrap, recovery and depth-scan tables equal the cached
PREPUB: ones in `modeling_SFCVM/gf/` (same seeds; run of 2026-10-03). 08a's Fig-2 products read the
PREPUB: cached bootstrap table, so this check is what ties them to this notebook.

**Inputs**: 08a's products (`results_comparison<tag>.csv` for the selected sigmas) and everything
08a reads; from `modeling_SFCVM/gf/`: the sensitivity runs (`final_comparison.csv`,
`results_comparison<tag>.csv`, `inversion_products<tag>.npz`, `lcurve_<medium><tag>.csv`) and the
cached `uncertainty_*`, `recovery_*` and `depth_scan_*` tables of the checks.
**Outputs** (`outputs/08b_3d_model_tests/`): `uncertainty_<run>_<medium>.csv`, `recovery_<run>_<medium>.csv`,
`recovery_profiles.csv`, `depth_scan_<run>_<medium>.csv`, `sensitivity_runs.csv`, `tests_summary.txt`,
the stats sidecar `08b_3d_model_tests_stats.txt`.
**Figures**: `qc08b_bootstrap`, `figS_pylith_recovery`, `figS_pylith_depth_scan`,
`qc08b_summary_<run>_<medium>` (one page per run), `figS_pylith_all_tests`.
**Registry keys** `08b_*`.

**What to check.** The selected model's bootstrap Mw30 2-sigma is small next to the sensitivity
range (`08b_sens_*`); the synthetic recovery shows no one-sided west/east mean residual larger
than the noise sd; every depth-scan amplitude is positive (asserted); the all-tests figure draws
as many runs as the table has (asserted).

In [1]:
## [1]
# ------------------------
# 08a's cells [1]-[10], then this notebook's own paths
# ------------------------
import ast
import glob
import os
import re
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from TulePond_filepaths import common_paths, repo_dir, paths_out, paths_sfcvm
import insar_utils as utils
from insar_utils import mpl_paper_style

t_setup = time.time()
source_08a = open(os.path.join(repo_dir, "scripts", "08a_3d_model.py")).read()
head_08a = source_08a.split("# %% [markdown]\n# ## [11]")[0]                 # cells [1]-[10]: G, 07a's machinery, invert_tents, numbers
assert "def invert_tents" in head_08a and "def potency_m3" in head_08a and "def numbers" in head_08a, "08a layout changed: cell [10] not found"
# 08a's own Registry line is replaced; 08a [7] quotes the same text inside a string (for 07a), which the ^...$ anchors skip
reg_line = re.compile(r"^REG = utils\.Registry\(notebook, common_paths, repo_dir\)$", re.M)
assert len(reg_line.findall(head_08a)) == 1, "08a layout changed: the Registry line"
head_08a = reg_line.sub("REG = None", head_08a)
ns08 = {"__name__": "nb08a"}
exec(head_08a, ns08)
for name_08a, value_08a in ns08.items():
    if name_08a not in {"notebook", "out_dir", "REG"} and not name_08a.startswith("__"):
        globals()[name_08a] = value_08a
print(f"08a cells [1]-[10] exec'd in {time.time() - t_setup:.0f} s; runs {RUNS}; selected {SELECTED}")

notebook = "08b_3d_model_tests"
out_dir = paths_out[notebook]
os.makedirs(out_dir, exist_ok=True)
dir_08a = paths_out["08a_3d_model"]
gf_cached = paths_sfcvm["gf"]                                                  # the earlier PyLith driver run's products (sensitivity runs live here)
REG = utils.Registry(notebook, common_paths, repo_dir)

tests_src = open(os.path.join(repo_dir, "scripts", "07b_uniform_model_tests.py")).read()


def param(name):
    """A parameter list/number from 07b's source (derived, not re-typed): the literal on the line `<name> = ...`."""
    match = re.search(rf"^{name}\s*=\s*(\[[^\]]*\]|[0-9.]+)", tests_src, re.S | re.M)
    assert match, f"{name} not found in 07b_uniform_model_tests.py"
    return ast.literal_eval(match.group(1))


recovery_depths = param("recovery_depths")
scan_taper = param("scan_taper")
scan_depths = param("scan_depths")
scan_km = (creep_lo_km + scan_taper, creep_hi_km - scan_taper)      # 07b [1]: SSE section tapered at its ends
print(f"07b parameters: n_boot {n_boot}, recovery depths {recovery_depths}, scan window km {scan_km[0]:.1f}-{scan_km[1]:.1f} "
      f"(taper {scan_taper} km), {len(scan_depths)} scan depths")


def selected_sigma(medium, cd):
    """The L-curve pick of 08a for this model (its results csv; 08a must have run)."""
    results_file = os.path.join(dir_08a, f"results_comparison{CD_TAG[cd]}.csv")
    assert os.path.exists(results_file), f"missing {results_file}: run 08a_3d_model first"
    return float(pd.read_csv(results_file).set_index("model").loc[f"PyLith {medium}", "sigma"])


in_window = np.clip(np.minimum(imp_km - scan_km[0], scan_km[1] - imp_km) / scan_taper + 1, 0, 1)   # 07b [9]
REPORT = ["Mw_30gpa", "Mw_sfcvm", "potency_m3", "surface_pct", "peak_slip_mm", "peak_depth_m"]


def check_table(mine, cached_file, cols, key_cols, rtol=1e-6):
    """CHECK: the columns `cols` of this run's table equal the cached product of the earlier run, row-matched on
    key_cols (surface % included: one denominator everywhere, 08a [10])."""
    assert os.path.exists(cached_file), f"missing cached product {cached_file}"
    theirs = pd.read_csv(cached_file)
    mine_sorted = mine.set_index(key_cols).sort_index()
    theirs_sorted = theirs.set_index(key_cols).sort_index()
    assert mine_sorted.index.equals(theirs_sorted.index), f"{os.path.basename(cached_file)}: different rows"
    worst = 0.0
    for col in cols:
        cached_values = theirs_sorted[col].values
        diff = np.abs(mine_sorted[col].values - cached_values) / np.maximum(np.abs(cached_values), 1e-9)
        worst = max(worst, float(np.nanmax(diff)))
    assert worst < rtol, f"{os.path.basename(cached_file)}: max rel diff {worst:.2e}"
    print(f"PASS: equals {os.path.basename(cached_file)} (max rel diff {worst:.1e})")

selected model: default_nuggetcd / sfcvm; companion default / sfcvm; PyLith products from /Users/dlindsay/work/proj_TulePond_DISP/modeling_SFCVM (TULEPOND_RUN_PYLITH=1)
# /Users/dlindsay/tools/internal/TulePond_DISP/drafts/pylith/generate_gmsh_vertical.py 2026-09-29 23:33
preset default: {'dx_fine': 100.0, 'dx_fault': 250.0, 'dx_surface': 250.0, 'dx_far': 6000.0, 'bias': 1.15}
tets 192634, nodes 36094
fault triangles 4535, fault vertices 2364
fault edge length (m): patch median 101 (min 74, max 125); outside median 192
tet minSICN: min 0.318, 1st pct 0.649, median 0.878, n<0.1 0, n<=0 0
tet gamma: min 0.329, median 0.857



wrote qc08a_mesh_fault.png
sfcvm_grid.py 2026-09-29 23:53 (aligned); SFCVM v26-0 /Users/dlindsay/work/proj_HaywardCreep/Inputs/USGS_SFCVM_v26-0/USGS_SFCVM_v26-0_detailed.h5
2386 output points; grid 165 x 197 x 80, vs floor 150 m/s, nu cap 0.4
X-X': trace point km 58.7 at (-794, 462) m; boundary offset b = 135 m


wrote /Users/dlindsay/work/proj_TulePond_DISP/Figures_2026-10-06/figS_pylith_materials_sfcvm_v26.png, /Users/dlindsay/work/proj_TulePond_DISP/Figures_2026-10-06/qc08a_materials_boundary_offset.png
gf_default_homog: PyLith 5.0.2, 2026-09-29T23:47:58.627614, arm64
gf_default_sfcvm: PyLith 5.0.2, 2026-09-30T00:19:55.367205, arm64
G: 2386 samples x 2105 impulses (of 2364 fault vertices); max |G| homog 0.552, max |G| sfcvm 0.822 m/m
sfcvm: PyLith cell mu vs grid at centroids (40630 cells in the fine box, top 1.5 km): median |rel diff| 0.000, 90th pct 0.003; PyLith mu range 0.072-64.5 GPa
sfcvm: fault-adjacent mu, top 100 m median: west 0.499, east 1.320 GPa
PASS: PyLith used the aligned SFCVM rigidity
SSE section: km 56.3-62.2 (04 section_lengths.csv)
dropped 1 non-numeric placemark(s): ['multi-splay zone']


mesh smooth12: 1510 nodes, 2610 triangles, 24.6 km of fault, dip 90


fault-parallel field at strike 146.51 deg: 74783 pixels; levelling plane +0.48 mm, +0.087, -0.115 mm/km


nugget_mm     0.790
short_mm      0.865
short_km      1.050
long_mm       0.000
long_km      50.000
quadtree: 2386 samples
mean event of 5 Tule Pond events: amp_2017.099, amp_2019.186, amp_2021.107, amp_2023.080, amp_2024.984
surface constraint: 57 profile points; surface % denominator: median of the measured crossings in the SSE section 8.23 mm
07a cells [1]-[6] exec'd in 10 s: 2386 samples, sigma grid [4, 6, 8, 12, 16, 20, 24, 32, 40, 48, 64, 96, 128], lambda (0.1, 0.05) km, surface floor 0.5 mm, n_boot 24
07a baseline: sigma 8, rms 2.148 mm, Mw 3.596
samples on opposite sides of the mesh fault and of 07a's trace: 6 (largest distance from the mesh fault 15 m); |mesh - 07a| signed distance: median 0.9 m, max 81.1 m


validation (a) PyLith homog vs CSI: {'max_abs_diff_mm': 10.0919, 'rms_diff_mm': 0.265, 'rms_signal_mm': 2.1772, 'rms_diff_near300_mm': 0.2959, 'rms_diff_far_mm': 0.0161, 'potency_pylith_m3': 39022.3629, 'potency_csi_m3': 39258.6453, 'rms_diff_okside_mm': 0.0896, 'rel_rms': 0.1217, 'rel_rms_okside': 0.0411}
validation (b) PyLith homog vs 2D screw dislocation: {'n': 879, 'D_eff_m': 1050.7196, 'rms_diff_mm': 0.2158, 'max_abs_diff_mm': 0.488, 'rms_signal_mm': 4.479, 'rel_rms': 0.0482}
wrote figS_pylith_validation.png
fwd_default_homog: slip -0.00..10.00 mm on 482 vertices; max |u| 5.56 mm
fwd_fine_homog: slip -0.00..10.00 mm on 1273 vertices; max |u| 5.46 mm
validation (c) superposition, G[homog] x slip vs forward run fwd_default_homog: {'max_abs_diff_mm': 1e-06, 'rms_diff_mm': 0.0, 'rms_signal_mm': 2.200818}
PASS: G x slip reproduces the forward run (homogeneous)


validation (d) forward default mesh vs CSI (same slip, 2380 samples on agreeing sides): {'rms_diff_mm': 0.0907, 'rms_diff_near300_mm': 0.101, 'rms_diff_far_mm': 0.0173, 'max_abs_diff_mm': 1.7136}
validation (d) forward fine mesh vs CSI (same slip, 2380 samples on agreeing sides): {'rms_diff_mm': 0.0768, 'rms_diff_near300_mm': 0.0856, 'rms_diff_far_mm': 0.0111, 'max_abs_diff_mm': 1.213}
validation (d) fine - default forward: rms 0.0827 mm, within 300 m 0.0921 mm, farther 0.0132 mm
wrote qc08a_validation_mesh.png


CSI TriangularTents: 2105 tents, 4048 faces (7 s)
surface % denominator: median of 58 measured crossings in the SSE section 8.23 mm (= 07a's crossing_median)
surface constraint: 89 crossings inside the mesh top edge (km 54.3-64.1; 35 beyond its ends), distance to the edge median 0.6 m, max 15.8 m; error median 0.51 mm
08a cells [1]-[10] exec'd in 54 s; runs [('sfcvm', 'nugget'), ('sfcvm', 'full'), ('homog', 'nugget'), ('homog', 'full')]; selected ('sfcvm', 'nugget')
Registry for 08b_3d_model_tests: run 7b4ec343, git 8d8e45f+
07b parameters: n_boot 24, recovery depths [0.3, 0.5, 0.8, 1.0, 1.5, 2.0], scan window km 56.8-61.7 (taper 0.5 km), 30 scan depths


## [2] Bootstrap and sigma sensitivity (07a [11])

n_boot noise realisations from each run's own data and surface covariances (Cholesky) added
to the model's own synthetics, re-inverted at the selected sigma, each on a fresh fault and
fresh copies of the data (seed 11, as 07a). The spread is the formal error of THIS
configuration; the structural range is the sensitivity table of [5]. Then the neighbouring grid
sigmas on the real data.

In [2]:
## [2]
rng = np.random.default_rng(11)                                   # 07a [11]'s seed
uncertainty = {}
for medium, cd in RUNS:
    sigma = selected_sigma(medium, cd)
    ref_fault, ref_data, ref_surface = invert_tents(medium, sigma, cd)
    ref = numbers(ref_fault, ref_data, ref_surface)
    chol_insar = np.linalg.cholesky(ref_data.Cd + 1e-9 * np.eye(ref_data.Cd.shape[0]))
    chol_surface = np.linalg.cholesky(ref_surface.Cd + 1e-9 * np.eye(ref_surface.Cd.shape[0]))
    synth_insar = ref_data.synth.copy()
    synth_surface = ref_surface.synth.copy()
    rows = []
    t0 = time.time()
    for k in range(n_boot):
        vel_i = synth_insar + np.dot(chol_insar, rng.standard_normal(synth_insar.size))
        vel_s = synth_surface + np.dot(chol_surface, rng.standard_normal(synth_surface.size))
        fault_model, insar_data, surface = invert_tents(medium, sigma, cd, vel_insar=vel_i, vel_surface=vel_s)
        rows.append(dict(kind="boot", k=k, sigma=sigma, **numbers(fault_model, insar_data, surface)))
        print(f"  {medium} {cd} boot {k + 1}/{n_boot}: Mw30 {rows[-1]['Mw_30gpa']:.3f}, surface {rows[-1]['surface_pct']:.0f} %")
    boot = pd.DataFrame(rows)
    boot_mean = {}
    boot_2sd = {}
    for col in REPORT:
        boot_mean[col] = float(boot[col].mean())
        boot_2sd[col] = float(2 * boot[col].std(ddof=1))
    summary = [dict(kind="reference", k=-1, sigma=sigma, **ref),
               dict(kind="boot_mean", k=-1, sigma=sigma, **boot_mean),
               dict(kind="boot_2sd", k=-1, sigma=sigma, **boot_2sd)]
    i_sig = sigma_grid.index(sigma)
    for j in (i_sig - 1, i_sig + 1):
        if 0 <= j < len(sigma_grid):
            sigma_neighbour = sigma_grid[j]
            fault_model, insar_data, surface = invert_tents(medium, sigma_neighbour, cd)
            summary.append(dict(kind="sigma_neighbour", k=-1, sigma=sigma_neighbour, **numbers(fault_model, insar_data, surface)))
    table = pd.concat([pd.DataFrame(summary), boot], ignore_index=True)
    table["provenance"] = f"08b_3d_model_tests {time.strftime('%Y-%m-%d %H:%M')}; G {run_names[medium]}; Cd {cd}; seed 11"
    out = os.path.join(out_dir, f"uncertainty_{run_tag(cd)}_{medium}.csv")
    table.to_csv(out, index=False)
    uncertainty[(medium, cd)] = table
    report_parts = []
    for col in REPORT:
        report_parts.append(f"{col} {boot_mean[col]:.3g} +/- {boot_2sd[col]:.2g}")
    print(f"{medium} {cd} (sigma {sigma:g}, {n_boot} realisations, {time.time() - t0:.0f} s): "
          + ", ".join(report_parts) + f"; wrote {os.path.basename(out)}")
    check_table(table, os.path.join(gf_cached, f"uncertainty_{run_tag(cd)}_{medium}.csv"), REPORT, ["kind", "k", "sigma"])   # PREPUB

plt.rcParams.update({"font.size": 12, "savefig.facecolor": "white", "figure.facecolor": "white"})
fig, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True)
for ax, (medium, cd) in zip(axes.ravel(), RUNS):
    table = uncertainty[(medium, cd)]
    boot = table[table.kind == "boot"]
    reference = table[table.kind == "reference"].iloc[0]
    ax.hist(boot.Mw_30gpa, bins=12, color=OK["blue"] if cd == "full" else OK["vermillion"], alpha=0.8)
    ax.axvline(reference.Mw_30gpa, color="k", lw=1.5, label=f"reference {reference.Mw_30gpa:.2f}")
    boot_mw_mean = boot.Mw_30gpa.mean()
    boot_mw_2sd = 2 * boot.Mw_30gpa.std(ddof=1)
    ax.axvspan(boot_mw_mean - boot_mw_2sd, boot_mw_mean + boot_mw_2sd, color=OK["grey"], alpha=0.25,
               label=f"mean {boot_mw_mean:.2f} +/- {boot_mw_2sd:.2f} (2 sd)")
    for _, row in table[table.kind == "sigma_neighbour"].iterrows():
        ax.axvline(row.Mw_30gpa, color=OK["orange"], ls="--", lw=1.2, label=f"sigma {row.sigma:g}: {row.Mw_30gpa:.2f}")
    ax.set_xlabel("Mw (30 GPa)")
    ax.set_ylabel("realisations")
    ax.legend(fontsize=9)
    ax.set_title(f"{medium}, Cd {cd}: bootstrap ({n_boot}) and neighbouring sigmas", loc="left", fontsize=12)
fig.savefig(os.path.join(fig_dir, "qc08b_bootstrap.png"), dpi=300)
plt.close(fig)
print("wrote qc08b_bootstrap.png")

---------------------------------
---------------------------------
Assembling d for fault sfcvm nugget sigma 12.0
Dealing with data Downsampled fault-parallel of type insar
Dealing with data crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling Cd for fault sfcvm nugget sigma 12.0
Dealing with data Downsampled fault-parallel of type insar
Dealing with data crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling G for transformation ramps
Dealing with Downsampled fault-parallel of type insar
Dealing with crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling d for transformation ramps
Dealing with data Downsampled fault-parallel of type insar
Dealing with data crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling Cd for transformation ramps
Dealing with

---------------------------------
---------------------------------
Assembling d for fault sfcvm nugget sigma 12.0
Dealing with data Downsampled fault-parallel of type insar
Dealing with data crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling Cd for fault sfcvm nugget sigma 12.0
Dealing with data Downsampled fault-parallel of type insar
Dealing with data crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling G for transformation ramps
Dealing with Downsampled fault-parallel of type insar
Dealing with crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling d for transformation ramps
Dealing with data Downsampled fault-parallel of type insar
Dealing with data crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling Cd for transformation ramps
Dealing with

  sfcvm nugget boot 1/24: Mw30 3.775, surface 116 %
---------------------------------
---------------------------------
Assembling d for fault sfcvm nugget sigma 12.0
Dealing with data Downsampled fault-parallel of type insar
Dealing with data crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling Cd for fault sfcvm nugget sigma 12.0
Dealing with data Downsampled fault-parallel of type insar
Dealing with data crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling G for transformation ramps
Dealing with Downsampled fault-parallel of type insar
Dealing with crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling d for transformation ramps
Dealing with data Downsampled fault-parallel of type insar
Dealing with data crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------

  sfcvm nugget boot 2/24: Mw30 3.762, surface 118 %
---------------------------------
---------------------------------
Assembling d for fault sfcvm nugget sigma 12.0
Dealing with data Downsampled fault-parallel of type insar
Dealing with data crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling Cd for fault sfcvm nugget sigma 12.0
Dealing with data Downsampled fault-parallel of type insar
Dealing with data crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling G for transformation ramps
Dealing with Downsampled fault-parallel of type insar
Dealing with crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling d for transformation ramps
Dealing with data Downsampled fault-parallel of type insar
Dealing with data crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------

  sfcvm nugget boot 3/24: Mw30 3.768, surface 115 %
---------------------------------
---------------------------------
Assembling d for fault sfcvm nugget sigma 12.0
Dealing with data Downsampled fault-parallel of type insar
Dealing with data crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling Cd for fault sfcvm nugget sigma 12.0
Dealing with data Downsampled fault-parallel of type insar
Dealing with data crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling G for transformation ramps
Dealing with Downsampled fault-parallel of type insar
Dealing with crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling d for transformation ramps
Dealing with data Downsampled fault-parallel of type insar
Dealing with data crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------

  sfcvm nugget boot 4/24: Mw30 3.754, surface 115 %
---------------------------------
---------------------------------
Assembling d for fault sfcvm nugget sigma 12.0
Dealing with data Downsampled fault-parallel of type insar
Dealing with data crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling Cd for fault sfcvm nugget sigma 12.0
Dealing with data Downsampled fault-parallel of type insar
Dealing with data crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling G for transformation ramps
Dealing with Downsampled fault-parallel of type insar
Dealing with crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling d for transformation ramps
Dealing with data Downsampled fault-parallel of type insar
Dealing with data crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------

... (100 outputs left out)

  homog full boot 24/24: Mw30 3.371, surface 84 %
---------------------------------
---------------------------------
Assembling d for fault homog full sigma 8
Dealing with data Downsampled fault-parallel of type insar
Dealing with data crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling Cd for fault homog full sigma 8
Dealing with data Downsampled fault-parallel of type insar
Dealing with data crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling G for transformation ramps
Dealing with Downsampled fault-parallel of type insar
Dealing with crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling d for transformation ramps
Dealing with data Downsampled fault-parallel of type insar
Dealing with data crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling 

---------------------------------
---------------------------------
Assembling d for fault homog full sigma 16
Dealing with data Downsampled fault-parallel of type insar
Dealing with data crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling Cd for fault homog full sigma 16
Dealing with data Downsampled fault-parallel of type insar
Dealing with data crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling G for transformation ramps
Dealing with Downsampled fault-parallel of type insar
Dealing with crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling d for transformation ramps
Dealing with data Downsampled fault-parallel of type insar
Dealing with data crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling Cd for transformation ramps
Dealing with data Do

homog full (sigma 12, 24 realisations, 199 s): Mw_30gpa 3.39 +/- 0.02, Mw_sfcvm 2.54 +/- 0.06, potency_m3 5.14e+03 +/- 3.5e+02, surface_pct 83 +/- 4.3, peak_slip_mm 10.1 +/- 0.63, peak_depth_m 0 +/- 0; wrote uncertainty_default_homog.csv
PASS: equals uncertainty_default_homog.csv (max rel diff 4.4e-15)


wrote qc08b_bootstrap.png


## [3] Synthetic recovery (07b [11])

10 mm uniform right-lateral slip from the surface to D over the SSE section (07b's tapered
window), forward through the PyLith G of the medium, noise from the run's Cd (seed 7), the
surface constraint's own error on each measured crossing, inverted with the same setup at the
selected sigma. Reported: potency recovered / input, the depth below which the section-mean slip
falls under 10 % of its largest value over depth, and the mean residual within 300 m west / east
of the trace (an artefact flag: a one-sided bias means the medium cannot fit that side).

In [3]:
## [3]
recovery_rng = np.random.default_rng(7)                          # 07b's seed
in_scan_section = (imp_km >= scan_km[0]) & (imp_km <= scan_km[1])
zbins = np.arange(0, 3.01, 0.1)                                  # depth bins of the recovered profile (km)
recovery_tables = {}
profiles = []
for medium, cd in RUNS:
    sigma = selected_sigma(medium, cd)
    _, cd_data, _ = invert_tents(medium, sigma, cd)              # the run's Cd (InSAR)
    chol = np.linalg.cholesky(cd_data.Cd + 1e-9 * np.eye(cd_data.Cd.shape[0]))
    surf_ref, G_surf, _ = surface_constraint()
    noise_w = float(np.sqrt(np.mean(np.diag(cd_data.Cd)[west300])))
    noise_e = float(np.sqrt(np.mean(np.diag(cd_data.Cd)[east300])))
    rows = []
    for D in recovery_depths:
        slip_in = 10.0 * in_window * (imp_depth_km < D)
        pot_in = potency_m3(slip_in)
        vel_i = G[medium].dot(slip_in) + np.dot(chol, recovery_rng.standard_normal(cd_data.vel.size))
        vel_s = G_surf.dot(slip_in) + np.sqrt(np.diag(surf_ref.Cd)) * recovery_rng.standard_normal(surf_ref.vel.size)
        fault_model, insar_data, surface = invert_tents(medium, sigma, cd, vel_insar=vel_i, vel_surface=vel_s)
        num = numbers(fault_model, insar_data, surface)
        res = insar_data.vel - insar_data.synth
        slip_out = fault_model.slip[:, 0]
        prof = []                                                # section-mean recovered slip per 0.1 km depth bin
        for z0 in zbins[:-1]:
            in_bin = in_scan_section & (imp_depth_km >= z0) & (imp_depth_km < z0 + 0.1)
            if np.any(in_bin):
                prof.append(slip_out[in_bin].mean())
            else:
                prof.append(np.nan)
        prof = np.array(prof)
        below = np.nonzero(prof < 0.1 * np.nanmax(prof))[0]
        d_rec = float(zbins[below[0]]) if below.size else np.nan
        rows.append(dict(D_km=D, potency_in_m3=pot_in, potency_out_m3=num["potency_m3"], moment_recovery=num["potency_m3"] / pot_in,
                         Mw_in=float(moment_magnitude(mu_ref * pot_in)), Mw_out=num["Mw_30gpa"], depth_10pct_km=d_rec,
                         peak_out_mm=num["peak_slip_mm"], peak_depth_m=num["peak_depth_m"], surface_pct=num["surface_pct"],
                         rms_mm=num["rms_mm"], rms_west300_mm=num["rms_west300_mm"], rms_east300_mm=num["rms_east300_mm"],
                         mean_res_west300_mm=float(res[west300].mean()), mean_res_east300_mm=float(res[east300].mean()),
                         noise_sd_west300_mm=noise_w, noise_sd_east300_mm=noise_e))
        input_profile = []
        for z0 in zbins[:-1]:
            input_profile.append(10.0 * float(z0 + 0.05 < D))
        profiles.append(pd.DataFrame(dict(run=run_tag(cd), medium=medium, D_km=D, z_top_km=zbins[:-1], slip_mm=prof,
                                          input_mm=input_profile)))
        print(f"  {medium} {cd} input 10 mm to {D:g} km: moment recovered {rows[-1]['moment_recovery']:.2f}, Mw {rows[-1]['Mw_in']:.2f} -> "
              f"{rows[-1]['Mw_out']:.2f}, depth(10 %) {d_rec:.1f} km, W/E rms {rows[-1]['rms_west300_mm']:.2f}/{rows[-1]['rms_east300_mm']:.2f} "
              f"(noise sd {noise_w:.2f}/{noise_e:.2f}), W/E mean residual {rows[-1]['mean_res_west300_mm']:+.2f}/{rows[-1]['mean_res_east300_mm']:+.2f} mm")
    table = pd.DataFrame(rows)
    table["provenance"] = f"08b_3d_model_tests {time.strftime('%Y-%m-%d %H:%M')}; G {run_names[medium]}; Cd {cd}; sigma {sigma:g}; seed 7"
    table.to_csv(os.path.join(out_dir, f"recovery_{run_tag(cd)}_{medium}.csv"), index=False)
    recovery_tables[(medium, cd)] = table
    check_table(table, os.path.join(gf_cached, f"recovery_{run_tag(cd)}_{medium}.csv"), ["moment_recovery", "Mw_out"], ["D_km"])   # PREPUB
pd.concat(profiles).to_csv(os.path.join(out_dir, "recovery_profiles.csv"), index=False)

import plot_tests_figures                                  # the recovery figure, drawn from the products just written
plot_tests_figures.main(out_dir, os.path.join(fig_dir, "figS_pylith_recovery.png"))

---------------------------------
---------------------------------
Assembling d for fault sfcvm nugget sigma 12.0
Dealing with data Downsampled fault-parallel of type insar
Dealing with data crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling Cd for fault sfcvm nugget sigma 12.0
Dealing with data Downsampled fault-parallel of type insar
Dealing with data crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling G for transformation ramps
Dealing with Downsampled fault-parallel of type insar
Dealing with crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling d for transformation ramps
Dealing with data Downsampled fault-parallel of type insar
Dealing with data crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling Cd for transformation ramps
Dealing with

---------------------------------
---------------------------------
Assembling d for fault sfcvm nugget sigma 12.0
Dealing with data Downsampled fault-parallel of type insar
Dealing with data crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling Cd for fault sfcvm nugget sigma 12.0
Dealing with data Downsampled fault-parallel of type insar
Dealing with data crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling G for transformation ramps
Dealing with Downsampled fault-parallel of type insar
Dealing with crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling d for transformation ramps
Dealing with data Downsampled fault-parallel of type insar
Dealing with data crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling Cd for transformation ramps
Dealing with

  sfcvm nugget input 10 mm to 0.3 km: moment recovered 1.09, Mw 3.73 -> 3.75, depth(10 %) 0.7 km, W/E rms 0.72/0.74 (noise sd 0.70/0.71), W/E mean residual -0.08/+0.15 mm
---------------------------------
---------------------------------
Assembling d for fault sfcvm nugget sigma 12.0
Dealing with data Downsampled fault-parallel of type insar
Dealing with data crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling Cd for fault sfcvm nugget sigma 12.0
Dealing with data Downsampled fault-parallel of type insar
Dealing with data crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling G for transformation ramps
Dealing with Downsampled fault-parallel of type insar
Dealing with crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling d for transformation ramps
Dealing with data Downsampled fault-parallel of type insar
D

  sfcvm nugget input 10 mm to 0.5 km: moment recovered 1.02, Mw 3.87 -> 3.87, depth(10 %) 1.2 km, W/E rms 0.73/0.77 (noise sd 0.70/0.71), W/E mean residual -0.11/+0.15 mm
---------------------------------
---------------------------------
Assembling d for fault sfcvm nugget sigma 12.0
Dealing with data Downsampled fault-parallel of type insar
Dealing with data crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling Cd for fault sfcvm nugget sigma 12.0
Dealing with data Downsampled fault-parallel of type insar
Dealing with data crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling G for transformation ramps
Dealing with Downsampled fault-parallel of type insar
Dealing with crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling d for transformation ramps
Dealing with data Downsampled fault-parallel of type insar
D

  sfcvm nugget input 10 mm to 0.8 km: moment recovered 0.88, Mw 4.01 -> 3.97, depth(10 %) 1.4 km, W/E rms 0.69/0.83 (noise sd 0.70/0.71), W/E mean residual -0.10/+0.22 mm
---------------------------------
---------------------------------
Assembling d for fault sfcvm nugget sigma 12.0
Dealing with data Downsampled fault-parallel of type insar
Dealing with data crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling Cd for fault sfcvm nugget sigma 12.0
Dealing with data Downsampled fault-parallel of type insar
Dealing with data crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling G for transformation ramps
Dealing with Downsampled fault-parallel of type insar
Dealing with crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling d for transformation ramps
Dealing with data Downsampled fault-parallel of type insar
D

  sfcvm nugget input 10 mm to 1 km: moment recovered 0.87, Mw 4.07 -> 4.03, depth(10 %) 1.7 km, W/E rms 0.72/0.83 (noise sd 0.70/0.71), W/E mean residual -0.04/+0.21 mm
---------------------------------
---------------------------------
Assembling d for fault sfcvm nugget sigma 12.0
Dealing with data Downsampled fault-parallel of type insar
Dealing with data crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling Cd for fault sfcvm nugget sigma 12.0
Dealing with data Downsampled fault-parallel of type insar
Dealing with data crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling G for transformation ramps
Dealing with Downsampled fault-parallel of type insar
Dealing with crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling d for transformation ramps
Dealing with data Downsampled fault-parallel of type insar
Dea

... (20 outputs left out)

  homog full input 10 mm to 1 km: moment recovered 0.18, Mw 4.07 -> 3.57, depth(10 %) 0.4 km, W/E rms 1.56/1.50 (noise sd 1.12/1.12), W/E mean residual -0.89/+0.50 mm
---------------------------------
---------------------------------
Assembling d for fault homog full sigma 12.0
Dealing with data Downsampled fault-parallel of type insar
Dealing with data crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling Cd for fault homog full sigma 12.0
Dealing with data Downsampled fault-parallel of type insar
Dealing with data crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling G for transformation ramps
Dealing with Downsampled fault-parallel of type insar
Dealing with crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling d for transformation ramps
Dealing with data Downsampled fault-parallel of type insar
Dealing w

  homog full input 10 mm to 1.5 km: moment recovered 0.12, Mw 4.19 -> 3.57, depth(10 %) 0.4 km, W/E rms 1.61/1.58 (noise sd 1.12/1.12), W/E mean residual -0.50/+0.79 mm
---------------------------------
---------------------------------
Assembling d for fault homog full sigma 12.0
Dealing with data Downsampled fault-parallel of type insar
Dealing with data crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling Cd for fault homog full sigma 12.0
Dealing with data Downsampled fault-parallel of type insar
Dealing with data crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling G for transformation ramps
Dealing with Downsampled fault-parallel of type insar
Dealing with crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling d for transformation ramps
Dealing with data Downsampled fault-parallel of type insar
Dealing

  homog full input 10 mm to 2 km: moment recovered 0.09, Mw 4.27 -> 3.57, depth(10 %) 0.4 km, W/E rms 1.12/1.93 (noise sd 1.12/1.12), W/E mean residual +0.04/+1.45 mm
PASS: equals recovery_default_homog.csv (max rel diff 6.9e-16)
wrote /Users/dlindsay/work/proj_TulePond_DISP/Figures_2026-10-06/figS_pylith_recovery.png


'/Users/dlindsay/work/proj_TulePond_DISP/Figures_2026-10-06/figS_pylith_recovery.png'

## [4] Fixed-depth scan (07b [9])

Uniform slip to depth D over the tapered SSE window, amplitude and a constant by whitened least
squares with the run's Cd on the ramp-removed data of the reference solution, chi2(D), the
bracket delta chi2 <= 4 and the Mw of each D. No smoothing, no sigma: the depth the data alone
prefer under each covariance. Every amplitude must be positive (asserted).

In [4]:
## [4]
scans = {}
for medium, cd in RUNS:
    _, cd_data, _ = invert_tents(medium, selected_sigma(medium, cd), cd)
    chol = np.linalg.cholesky(cd_data.Cd + 1e-9 * np.eye(cd_data.Cd.shape[0]))
    data_w = np.linalg.solve(chol, cd_data.vel)                   # the ramp-removed data of the reference solution
    rows = []
    for D in scan_depths:
        if D > imp_depth_km.max() + 1e-6:
            continue                                              # deeper than the fault
        shape = in_window * (imp_depth_km < D)
        design = np.c_[G[medium].dot(shape), np.ones(cd_data.vel.size)]
        design_w = np.linalg.solve(chol, design)
        coef, *_ = np.linalg.lstsq(design_w, data_w, rcond=None)
        resid_w = data_w - design_w.dot(coef)
        pot = potency_m3(shape * coef[0])
        assert pot > 0, f"depth scan {medium} {cd} D={D}: non-positive amplitude {coef[0]:.2f} mm"
        rows.append(dict(D_km=D, slip_mm=float(coef[0]), constant_mm=float(coef[1]), chi2=float(np.dot(resid_w, resid_w)),
                         potency_m3=pot, Mw=float(moment_magnitude(mu_ref * pot))))
    scan = pd.DataFrame(rows)
    scan["dchi2"] = scan.chi2 - scan.chi2.min()
    scan["provenance"] = f"08b_3d_model_tests {time.strftime('%Y-%m-%d %H:%M')}; G {run_names[medium]}; Cd {cd}"
    scan.to_csv(os.path.join(out_dir, f"depth_scan_{run_tag(cd)}_{medium}.csv"), index=False)
    scans[(medium, cd)] = scan
    best = scan.loc[scan.chi2.idxmin()]
    bracket = scan[scan.dchi2 <= 4]
    print(f"depth scan {medium} {cd}: best D {best.D_km:g} km ({best.slip_mm:.1f} mm, Mw {best.Mw:.2f}); delta chi2 <= 4: "
          f"{bracket.D_km.min():g}-{bracket.D_km.max():g} km, Mw {bracket.Mw.min():.2f}-{bracket.Mw.max():.2f}")
    check_table(scan, os.path.join(gf_cached, f"depth_scan_{run_tag(cd)}_{medium}.csv"), ["chi2", "slip_mm"], ["D_km"])   # PREPUB

mpl_paper_style(base=7)
fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.6), constrained_layout=True)
for ax, cd, letter in zip(axes, ("nugget", "full"), "AB"):
    for medium, colour in (("sfcvm", OK["vermillion"]), ("homog", OK["blue"])):
        scan = scans[(medium, cd)]
        ax.plot(scan.D_km, scan.dchi2, "o-", color=colour, ms=3, lw=1, label=f"{medium}")
        bracket = scan[scan.dchi2 <= 4]
        ax.axvspan(bracket.D_km.min(), bracket.D_km.max(), color=colour, alpha=0.12)
    ax.axhline(4, color="0.5", lw=0.6, ls="--")
    ax.set_xscale("log")
    ax.set_ylim(-1, 60)
    ax.set_xlabel("uniform slip to depth D (km)")
    ax.set_ylabel("delta chi2")
    ax.legend(fontsize=6)
    ax.set_title(r"$\bf{" + letter + r"}$  " + f"fixed-depth scan, Cd {cd} (shaded: delta chi2 <= 4)", loc="left", fontsize=7)
fig.savefig(os.path.join(fig_dir, "figS_pylith_depth_scan.png"), dpi=300)
plt.close(fig)
print("wrote figS_pylith_depth_scan.png")

---------------------------------
---------------------------------
Assembling d for fault sfcvm nugget sigma 12.0
Dealing with data Downsampled fault-parallel of type insar
Dealing with data crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling Cd for fault sfcvm nugget sigma 12.0
Dealing with data Downsampled fault-parallel of type insar
Dealing with data crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling G for transformation ramps
Dealing with Downsampled fault-parallel of type insar
Dealing with crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling d for transformation ramps
Dealing with data Downsampled fault-parallel of type insar
Dealing with data crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling Cd for transformation ramps
Dealing with

depth scan sfcvm nugget: best D 0.4 km (11.0 mm, Mw 3.83); delta chi2 <= 4: 0.4-0.4 km, Mw 3.83-3.83
PASS: equals depth_scan_default_nuggetcd_sfcvm.csv (max rel diff 2.0e-16)
---------------------------------
---------------------------------
Assembling d for fault sfcvm full sigma 12.0
Dealing with data Downsampled fault-parallel of type insar
Dealing with data crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling Cd for fault sfcvm full sigma 12.0
Dealing with data Downsampled fault-parallel of type insar
Dealing with data crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling G for transformation ramps
Dealing with Downsampled fault-parallel of type insar
Dealing with crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling d for transformation ramps
Dealing with data Downsampled fault-parallel of type insar
D

depth scan sfcvm full: best D 0.95 km (6.8 mm, Mw 3.94); delta chi2 <= 4: 0.7-1.1 km, Mw 3.85-3.98
PASS: equals depth_scan_default_sfcvm.csv (max rel diff 0.0e+00)
---------------------------------
---------------------------------
Assembling d for fault homog nugget sigma 12.0
Dealing with data Downsampled fault-parallel of type insar
Dealing with data crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling Cd for fault homog nugget sigma 12.0
Dealing with data Downsampled fault-parallel of type insar
Dealing with data crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling G for transformation ramps
Dealing with Downsampled fault-parallel of type insar
Dealing with crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling d for transformation ramps
Dealing with data Downsampled fault-parallel of type insar
Dealing 

depth scan homog nugget: best D 0.4 km (9.1 mm, Mw 3.78); delta chi2 <= 4: 0.4-0.4 km, Mw 3.78-3.78
PASS: equals depth_scan_default_nuggetcd_homog.csv (max rel diff 2.0e-16)
---------------------------------
---------------------------------
Assembling d for fault homog full sigma 12.0
Dealing with data Downsampled fault-parallel of type insar
Dealing with data crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling Cd for fault homog full sigma 12.0
Dealing with data Downsampled fault-parallel of type insar
Dealing with data crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling G for transformation ramps
Dealing with Downsampled fault-parallel of type insar
Dealing with crossings f0.5 of type surfaceslip
---------------------------------
---------------------------------
Assembling d for transformation ramps
Dealing with data Downsampled fault-parallel of type insar
De

depth scan homog full: best D 0.85 km (6.7 mm, Mw 3.91); delta chi2 <= 4: 0.6-1.25 km, Mw 3.81-4.02
PASS: equals depth_scan_default_homog.csv (max rel diff 0.0e+00)
wrote figS_pylith_depth_scan.png


## [5] The sensitivity runs: one page per run, and all runs on one figure

The table `final_comparison.csv` (every run: settings, sigma, rms, near-fault rms west / east,
chi2, surface %, potency, Mw at 30 GPa and with the SFCVM rigidity, the bootstrap 2-sigma where
made) and the per-run products are read from `modeling_SFCVM/gf/`; the runs are NOT recomputed
(each needs its own PyLith Green's-function run: mesh `shallow`, the dipping meshes, order-2
elements, the swath-7 and coherence-0.7 sample sets, the +/- 50 m SFCVM grids).
CHECKS (fail loud on a stale table): the table is newer than 06's `fig2_inputs.npz` and `fp_maps.npz`;
every PyLith row (and every `results_comparison*.csv` the figures read) carries the current 06 field's
fingerprint and the locked sigma; the `default` and `default_nuggetcd` rows equal 08a's fresh
`results_comparison*.csv`. One-page diagnostic per run and medium (`qc08b_summary_<run>_<medium>`), then the
all-tests figure (`figS_pylith_all_tests`): near-fault fit both sides, west fit against Mw, and
the run-by-run ladder of the west-side rms, homogeneous vs SFCVM on the same settings.

In [5]:
## [5]
def two_decimals(value):
    """A table cell to two decimals ('' for a missing value)."""
    return "" if pd.isna(value) else f"{value:.2f}"


def no_decimals(value):
    """A table cell to the nearest unit ('' for a missing value)."""
    return "" if pd.isna(value) else f"{value:.0f}"


final_file = os.path.join(gf_cached, "final_comparison.csv")
assert os.path.exists(final_file), f"missing {final_file}"
for field_file in ("fig2_inputs.npz", "fp_maps.npz"):                      # stale table = made before the current 06 field
    field_path = os.path.join(field_dir, field_file)
    assert os.path.getmtime(final_file) > os.path.getmtime(field_path), \
        f"{final_file} ({time.ctime(os.path.getmtime(final_file))}) is older than 06's {field_file} " \
        f"({time.ctime(os.path.getmtime(field_path))}): re-run the driver inversions and final_comparison.py"
sensitivity = pd.read_csv(final_file)
assert {"run", "medium", "sigma", "sigma_lcurve_pick", "field_fingerprint", "rms_W300_mm", "Mw_30gpa", "surface_pct"} <= set(sensitivity.columns), \
    f"{final_file} lacks the sigma / field provenance columns: made by an old final_comparison.py"
pylith_rows = sensitivity[sensitivity.run != "07 CSI baseline"]
stale = pylith_rows[pylith_rows.field_fingerprint.astype(str) != field_fingerprint]
assert stale.empty, f"{len(stale)} sensitivity rows fit another 06 field (not {field_fingerprint}): {sorted(set(stale.run))}"
off_sigma = pylith_rows[~np.isclose(pylith_rows.sigma.astype(float), sigma_locked)]
assert off_sigma.empty, f"{len(off_sigma)} sensitivity rows not at the locked sigma {sigma_locked:g}: {sorted(set(off_sigma.run))}"
for cd in CD_TAG:                                                           # the default rows ARE 08a's models
    results_08a = pd.read_csv(os.path.join(dir_08a, f"results_comparison{CD_TAG[cd]}.csv")).set_index("model")
    for medium in media:
        row = pylith_rows[(pylith_rows.run == run_tag(cd)) & (pylith_rows.medium == medium)]
        assert len(row) == 1, f"{run_tag(cd)} / {medium}: {len(row)} rows in {final_file}"
        for col in ("Mw_30gpa", "potency_m3", "surface_pct", "rms_mm"):
            mine = float(results_08a.loc[f"PyLith {medium}", col])
            assert abs(float(row[col].iloc[0]) - mine) <= 1e-6 * abs(mine), f"{final_file} {run_tag(cd)} / {medium} {col} differs from 08a's"
print(f"PASS: {final_file} is newer than the 06 field, every PyLith row fits field {field_fingerprint} at sigma {sigma_locked:g}, "
      f"and its default rows equal 08a's")
sensitivity["provenance"] = f"copied by 08b_3d_model_tests {time.strftime('%Y-%m-%d %H:%M')} from {final_file}"
sensitivity.to_csv(os.path.join(out_dir, "sensitivity_runs.csv"), index=False)
show = sensitivity.drop(columns=["summary_png", "provenance", "field_fingerprint", "made"]).copy()
for col in ("rms_mm", "rms_W300_mm", "rms_E300_mm", "Mw_30gpa", "Mw_sfcvm", "Mw_30gpa_2sd", "Mw_sfcvm_2sd", "surface_pct_2sd"):
    show[col] = show[col].map(two_decimals)
for col in ("chi2", "surface_pct", "potency_m3"):
    show[col] = show[col].map(no_decimals)
show["sigma"] = show["sigma"].map("{:g}".format)
show["sigma_lcurve_pick"] = show["sigma_lcurve_pick"].map(lambda v: "" if pd.isna(v) else f"{v:g}")
show["sigma_corner"] = show["sigma_corner"].map(lambda v: "" if pd.isna(v) else f"{v:.1f}")
cols = list(show.columns)
print("| " + " | ".join(cols) + " |")
print("|" + "---|" * len(cols))
for row in show.itertuples(index=False):
    cells = []
    for value in row:
        cells.append(str(value))
    print("| " + " | ".join(cells) + " |")
print(f"\n{len(pylith_rows)} PyLith inversions on {pylith_rows.run.nunique()} run tags (+ the 07a CSI baseline row)")

import plot_run_summary
import plot_test_summary
tags = []
for results_file in glob.glob(os.path.join(gf_cached, "results_comparison*.csv")):
    tags.append(os.path.basename(results_file).replace("results_comparison", "").replace(".csv", ""))
tags = sorted(tags)
for tag in tags:                                                            # the figures read every table: none may be stale
    table_tag = pd.read_csv(os.path.join(gf_cached, f"results_comparison{tag}.csv"))
    pylith_tag = table_tag[table_tag.model.str.startswith("PyLith ")]
    assert (pylith_tag.field_fingerprint.astype(str) == field_fingerprint).all(), f"results_comparison{tag}.csv fits another 06 field"
    assert np.allclose(pylith_tag.sigma.astype(float), sigma_locked), f"results_comparison{tag}.csv is not at the locked sigma"
summary_pages = []
for tag in tags:
    media_tag = []
    for run_medium in plot_run_summary.load_run(tag, gf_cached)["media"]:
        if run_medium != "sfcvmasis":                                            # aligned SFCVM only (Dani)
            media_tag.append(run_medium)
    summary_pages += plot_run_summary.main(tag, media_tag, gf_dir=gf_cached, fig_dir=fig_dir, prefix="qc08b_summary")
all_runs = plot_test_summary.main(gf_cached, os.path.join(fig_dir, "figS_pylith_all_tests.png"))
assert len(all_runs) == len(pylith_rows), f"all-tests figure drew {len(all_runs)} runs, the table has {len(pylith_rows)}"
print(f"{len(summary_pages)} one-page summaries; all-tests figure on {len(all_runs)} runs")

PASS: /Users/dlindsay/work/proj_TulePond_DISP/modeling_SFCVM/gf/final_comparison.csv is newer than the 06 field, every PyLith row fits field fp_maps.npz:4ea68b7e6ad9 at sigma 12, and its default rows equal 08a's
| run | medium | mesh | dip | Cd | surface | points | sigma | rms_mm | rms_W300_mm | rms_E300_mm | chi2 | surface_pct | potency_m3 | Mw_30gpa | Mw_sfcvm | sigma_lcurve_pick | sigma_corner | sigma_rule | Mw_30gpa_2sd | Mw_sfcvm_2sd | surface_pct_2sd |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| 07 CSI baseline | homog (Meade TDE) | 07 smooth12 | 90 | full | on | default | 8 | 2.15 | 2.79 | 1.63 | 3951 | 91 |  | 3.60 |  |  |  | 07a L-curve pick |  |  |  |
| default | homog | default | 90 | full | on | default | 12 | 2.18 | 2.82 | 1.68 | 4126 | 94 | 7948 | 3.52 | 2.69 | 12 | 9.6 | fixed (PYLITH_SIGMA=locked) | 0.02 | 0.06 | 4.26 |
| default | sfcvm | default | 90 | full | on | default | 12 | 1.91 | 2.48 | 1.36 | 4107 | 91 | 9054 | 3.

wrote /Users/dlindsay/work/proj_TulePond_DISP/Figures_2026-10-06/qc08b_summary_default_homog.png


wrote /Users/dlindsay/work/proj_TulePond_DISP/Figures_2026-10-06/qc08b_summary_default_sfcvm.png


wrote /Users/dlindsay/work/proj_TulePond_DISP/Figures_2026-10-06/qc08b_summary_default_coh07_homog.png


wrote /Users/dlindsay/work/proj_TulePond_DISP/Figures_2026-10-06/qc08b_summary_default_coh07_sfcvm.png


wrote /Users/dlindsay/work/proj_TulePond_DISP/Figures_2026-10-06/qc08b_summary_default_coh07_nuggetcd_homog.png


... (32 outputs left out)

wrote /Users/dlindsay/work/proj_TulePond_DISP/Figures_2026-10-06/qc08b_summary_default_swath7_nuggetcd_sfcvm.png


wrote /Users/dlindsay/work/proj_TulePond_DISP/Figures_2026-10-06/qc08b_summary_default_w50_sfcvm_w50.png


wrote /Users/dlindsay/work/proj_TulePond_DISP/Figures_2026-10-06/qc08b_summary_default_w50_nuggetcd_sfcvm_w50.png


wrote /Users/dlindsay/work/proj_TulePond_DISP/Figures_2026-10-06/figS_pylith_all_tests.png 40 runs
40 one-page summaries; all-tests figure on 40 runs


## [6] Summary and registry

`08b_*` = the selected model; `08b_hom_*`, `08b_full_*`, `08b_full_hom_*` as in 08a. The
structural range keys (`08b_sens_*`) are over every SFCVM run WITH the surface constraint in the
sensitivity table (mesh, Cd treatments, swath, dip, order, alignment, coherence), all at the locked
sigma; surface % on the one measured-crossing denominator. NB: `08b_*_2sig` and 08a's `08a_*_2sig` are the same
bootstrap (this notebook's fresh one and the cached one 08a reads, checked equal in [2]).

In [6]:
## [6]
KEY_08B = {("sfcvm", "nugget"): "08b", ("homog", "nugget"): "08b_hom", ("sfcvm", "full"): "08b_full", ("homog", "full"): "08b_full_hom"}
lines = []
for medium, cd in RUNS:
    table = uncertainty[(medium, cd)]
    boot_mean_row = table[table.kind == "boot_mean"].iloc[0]
    boot_2sd_row = table[table.kind == "boot_2sd"].iloc[0]
    reference = table[table.kind == "reference"].iloc[0]
    neighbours = table[table.kind == "sigma_neighbour"]
    rec = recovery_tables[(medium, cd)]
    scan = scans[(medium, cd)]
    bracket = scan[scan.dchi2 <= 4]
    neighbour_text = []
    for _, row in neighbours.iterrows():
        neighbour_text.append(f"{row.sigma:g}: {row.Mw_30gpa:.2f}")
    lines.append(f"{run_tag(cd)} / {medium} (sigma {reference.sigma:g}): Mw30 {reference.Mw_30gpa:.2f} "
                 f"(boot {boot_mean_row.Mw_30gpa:.2f} +/- {boot_2sd_row.Mw_30gpa:.2f}; sigma {', '.join(neighbour_text)}), "
                 f"Mw_sfcvm {reference.Mw_sfcvm:.2f} +/- {boot_2sd_row.Mw_sfcvm:.2f}, "
                 f"surface {reference.surface_pct:.0f} +/- {boot_2sd_row.surface_pct:.0f} %, "
                 f"peak {reference.peak_slip_mm:.1f} +/- {boot_2sd_row.peak_slip_mm:.1f} mm; "
                 f"recovery: moment {rec.moment_recovery.min():.2f}-{rec.moment_recovery.max():.2f} of input over D {rec.D_km.min():g}-{rec.D_km.max():g} km, "
                 f"W mean residual {rec.mean_res_west300_mm.min():+.2f}..{rec.mean_res_west300_mm.max():+.2f} mm; "
                 f"depth scan: best {scan.loc[scan.chi2.idxmin()].D_km:g} km, bracket {bracket.D_km.min():g}-{bracket.D_km.max():g} km "
                 f"(Mw {bracket.Mw.min():.2f}-{bracket.Mw.max():.2f})")
sens_sfcvm = pylith_rows[(pylith_rows.medium.str.startswith("sfcvm")) & (pylith_rows.surface == "on")]
lines.append(f"sensitivity table: {len(pylith_rows)} PyLith inversions at sigma {sigma_locked:g} on 06 field {field_fingerprint}; "
             f"SFSVM with surface constraint ({len(sens_sfcvm)} runs): "
             f"Mw30 {sens_sfcvm.Mw_30gpa.min():.2f}-{sens_sfcvm.Mw_30gpa.max():.2f}, rms W300 {sens_sfcvm.rms_W300_mm.min():.2f}-{sens_sfcvm.rms_W300_mm.max():.2f} mm, "
             f"surface {sens_sfcvm.surface_pct.min():.0f}-{sens_sfcvm.surface_pct.max():.0f} %; "
             f"own L-curve picks {', '.join(f'{v:g}' for v in sorted(set(sens_sfcvm.sigma_lcurve_pick)))} (not used)")
print("\n".join(lines))
with open(os.path.join(out_dir, "tests_summary.txt"), "w") as fh:
    fh.write(f"# 08b_3d_model_tests {time.strftime('%Y-%m-%d %H:%M')}\n" + "\n".join(lines) + "\n")
input_files = []
for cd in CD_TAG:
    input_files.append(os.path.join(dir_08a, f"results_comparison{CD_TAG[cd]}.csv"))
input_files.append(final_file)
for medium in media:
    input_files.append(os.path.join(paths_sfcvm["gf"], f"G_{run_names[medium]}.npz"))
# PREPUB-START: the cached tables of the earlier PyLith driver run that the checks compare with
for medium, cd in RUNS:
    for product in ("uncertainty", "recovery", "depth_scan"):
        input_files.append(os.path.join(gf_cached, f"{product}_{run_tag(cd)}_{medium}.csv"))
# PREPUB-END
REG.inputs(*input_files)
sidecar = REG.sidecar("08b_3d_model_tests", lines)
for medium, cd in RUNS:
    key = KEY_08B[(medium, cd)]
    table = uncertainty[(medium, cd)]
    boot_mean_row = table[table.kind == "boot_mean"].iloc[0]
    boot_2sd_row = table[table.kind == "boot_2sd"].iloc[0]
    neighbours = table[table.kind == "sigma_neighbour"]
    rec = recovery_tables[(medium, cd)]
    scan = scans[(medium, cd)]
    bracket = scan[scan.dchi2 <= 4]
    cell = "selected" if (medium, cd) == SELECTED else "comparison"
    REG.register(f"{key}_mw_30gpa_boot_mean", boot_mean_row.Mw_30gpa, "Mw", 2, where="M8", cell=cell, sidecar=sidecar)
    REG.register(f"{key}_mw_30gpa_2sig", boot_2sd_row.Mw_30gpa, "Mw", 2, where="M8", cell=cell, sidecar=sidecar)
    REG.register(f"{key}_mw_sfcvm_2sig", boot_2sd_row.Mw_sfcvm, "Mw", 2, where="M8", cell=cell, sidecar=sidecar)
    REG.register(f"{key}_surface_pct_2sig", boot_2sd_row.surface_pct, "%", 0, where="M8", cell=cell, sidecar=sidecar)
    REG.register(f"{key}_potency_2sig_m3", boot_2sd_row.potency_m3, "m3", 0, where="M8", cell=cell, sidecar=sidecar)
    REG.register(f"{key}_mw_30gpa_sigma_lo", float(neighbours.Mw_30gpa.min()), "Mw", 2, where="M8", cell="sigma neighbours", sidecar=sidecar)
    REG.register(f"{key}_mw_30gpa_sigma_hi", float(neighbours.Mw_30gpa.max()), "Mw", 2, where="M8", cell="sigma neighbours", sidecar=sidecar)
    REG.register(f"{key}_recovery_ratio_min", float(rec.moment_recovery.min()), "", 2, where="M8", cell="recovery", sidecar=sidecar)
    REG.register(f"{key}_recovery_ratio_max", float(rec.moment_recovery.max()), "", 2, where="M8", cell="recovery", sidecar=sidecar)
    REG.register(f"{key}_recovery_west_bias_max_mm", float(rec.mean_res_west300_mm.abs().max()), "mm", 2, where="M8", cell="recovery", sidecar=sidecar)
    REG.register(f"{key}_depth_best_km", float(scan.loc[scan.chi2.idxmin()].D_km), "km", 2, where="M8", cell="depth scan", sidecar=sidecar)
    REG.register(f"{key}_depth_bracket_lo_km", float(bracket.D_km.min()), "km", 2, where="M8", cell="depth scan", sidecar=sidecar)
    REG.register(f"{key}_depth_bracket_hi_km", float(bracket.D_km.max()), "km", 2, where="M8", cell="depth scan", sidecar=sidecar)
    REG.register(f"{key}_depth_bracket_mw_lo", float(bracket.Mw.min()), "Mw", 2, where="M8", cell="depth scan", sidecar=sidecar)
    REG.register(f"{key}_depth_bracket_mw_hi", float(bracket.Mw.max()), "Mw", 2, where="M8", cell="depth scan", sidecar=sidecar)
REG.register("08b_n_sensitivity_runs", int(len(pylith_rows)), "count", 0, where="M8", cell="sensitivity", sidecar=sidecar)
REG.register("08b_sens_mw_30gpa_min", float(sens_sfcvm.Mw_30gpa.min()), "Mw", 2, where="M8", cell="sensitivity", sidecar=sidecar)
REG.register("08b_sens_mw_30gpa_max", float(sens_sfcvm.Mw_30gpa.max()), "Mw", 2, where="M8", cell="sensitivity", sidecar=sidecar)
REG.register("08b_sens_surface_pct_min", float(sens_sfcvm.surface_pct.min()), "%", 0, where="M8", cell="sensitivity", sidecar=sidecar)
REG.register("08b_sens_surface_pct_max", float(sens_sfcvm.surface_pct.max()), "%", 0, where="M8", cell="sensitivity", sidecar=sidecar)
REG.render()
print("08b_3d_model_tests complete")

default_nuggetcd / sfcvm (sigma 12): Mw30 3.79 (boot 3.77 +/- 0.01; sigma 8: 3.73, 16: 3.82), Mw_sfcvm 3.10 +/- 0.06, surface 114 +/- 3 %, peak 16.4 +/- 0.6 mm; recovery: moment 0.69-1.09 of input over D 0.3-2 km, W mean residual -0.11..-0.03 mm; depth scan: best 0.4 km, bracket 0.4-0.4 km (Mw 3.83-3.83)
default / sfcvm (sigma 12): Mw30 3.56 (boot 3.40 +/- 0.03; sigma 8: 3.46, 16: 3.62), Mw_sfcvm 2.83 +/- 0.11, surface 91 +/- 4 %, peak 10.7 +/- 0.5 mm; recovery: moment 0.11-0.53 of input over D 0.3-2 km, W mean residual -0.43..+0.07 mm; depth scan: best 0.95 km, bracket 0.7-1.1 km (Mw 3.85-3.98)
default_nuggetcd / homog (sigma 12): Mw30 3.73 (boot 3.75 +/- 0.02; sigma 8: 3.70, 16: 3.75), Mw_sfcvm 3.01 +/- 0.06, surface 105 +/- 3 %, peak 17.9 +/- 0.5 mm; recovery: moment 0.54-1.15 of input over D 0.3-2 km, W mean residual -0.14..+0.02 mm; depth scan: best 0.4 km, bracket 0.4-0.4 km (Mw 3.78-3.78)
default / homog (sigma 12): Mw30 3.52 (boot 3.39 +/- 0.02; sigma 8: 3.43, 16: 3.58), Mw_sfc

  registered 08b_mw_30gpa_2sig = 0.01 Mw  -> M8
  registered 08b_mw_sfcvm_2sig = 0.06 Mw  -> M8
  registered 08b_surface_pct_2sig = 3 %  -> M8
  registered 08b_potency_2sig_m3 = 963 m3  -> M8


  registered 08b_mw_30gpa_sigma_lo = 3.73 Mw  -> M8
  registered 08b_mw_30gpa_sigma_hi = 3.82 Mw  -> M8
  registered 08b_recovery_ratio_min = 0.69   -> M8


  registered 08b_recovery_ratio_max = 1.09   -> M8
  registered 08b_recovery_west_bias_max_mm = 0.11 mm  -> M8
  registered 08b_depth_best_km = 0.40 km  -> M8
  registered 08b_depth_bracket_lo_km = 0.40 km  -> M8


  registered 08b_depth_bracket_hi_km = 0.40 km  -> M8
  registered 08b_depth_bracket_mw_lo = 3.83 Mw  -> M8
  registered 08b_depth_bracket_mw_hi = 3.83 Mw  -> M8
  registered 08b_full_mw_30gpa_boot_mean = 3.40 Mw  -> M8


  registered 08b_full_mw_30gpa_2sig = 0.03 Mw  -> M8
  registered 08b_full_mw_sfcvm_2sig = 0.11 Mw  -> M8
  registered 08b_full_surface_pct_2sig = 4 %  -> M8
  registered 08b_full_potency_2sig_m3 = 464 m3  -> M8


... (9 outputs left out)

  registered 08b_full_hom_depth_best_km = 0.85 km  -> M8
  registered 08b_full_hom_depth_bracket_lo_km = 0.60 km  -> M8
  registered 08b_full_hom_depth_bracket_hi_km = 1.25 km  -> M8
  registered 08b_full_hom_depth_bracket_mw_lo = 3.81 Mw  -> M8


  registered 08b_full_hom_depth_bracket_mw_hi = 4.02 Mw  -> M8
  registered 08b_n_sensitivity_runs = 40 count  -> M8
  registered 08b_sens_mw_30gpa_min = 3.45 Mw  -> M8
  registered 08b_sens_mw_30gpa_max = 3.82 Mw  -> M8


  registered 08b_sens_surface_pct_min = 76 %  -> M8
  registered 08b_sens_surface_pct_max = 134 %  -> M8
  01_prep_inputs: its latest run registered no numbers (sidecar newer than its last registry row); its keys are retired


Rendered registry to: /Users/dlindsay/work/proj_TulePond_DISP/outputs/registry.txt
08b_3d_model_tests complete
